# 6 — Reducción No Lineal y Visualización (UMAP / t-SNE)

Los notebooks anteriores no muestran cómo se distribuyen las muestras en el espacio de features. Este notebook responde:

- ¿Son las clases de temperatura linealmente separables?
- ¿Dónde están las confusiones reales (clases solapadas)?
- ¿Qué familias de features generan clusters más compactos?
- ¿Existe estructura de gradiente continuo (temperatura como manifold 1D)?

**Técnicas:**
- **UMAP** (Uniform Manifold Approximation and Projection): rápido, preserva estructura global y local
- **t-SNE**: referencia, énfasis en estructura local

**Visualizaciones:**
1. UMAP coloreado por temperatura (gradiente continuo)
2. UMAP coloreado por familia de features
3. t-SNE vs UMAP comparativa
4. Densidad por rango térmico

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

NSAMPLES = 3000  # subsample para velocidad

In [ ]:
df = pd.read_csv(CSV_PATH)

META_COLS = {
    "sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio",
}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

if FEAT_JSON.exists():
    with open(FEAT_JSON) as f: FEAT_COLS = json.load(f)
else:
    FEAT_COLS = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
print(f"Features: {len(FEAT_COLS)}")

# Subsample estratificado para velocidad
df_sub = (df.groupby("temperature", group_keys=False)
            .apply(lambda x: x.sample(min(NSAMPLES // len(df["temperature"].unique()), len(x)),
                                       random_state=42)))
print(f"Subsample: {len(df_sub)} muestras ({df_sub['temperature'].nunique()} temperaturas)")

# Preprocesado
imp = SimpleImputer(strategy="mean")
scaler = StandardScaler()
X = scaler.fit_transform(imp.fit_transform(df_sub[FEAT_COLS].values))
y_temp = df_sub["temperature"].values
y_rng  = df_sub["temperature"].apply(lambda t: "frio" if t<30 else ("templado" if t<=55 else "caliente")).values
print(f"X shape: {X.shape}")

In [ ]:
# ── PCA inicial para acelerar UMAP/tSNE ───────────────────────────────────────
print("PCA 50 componentes...")
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X)
print(f"Varianza explicada (50 PCs): {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# ── UMAP ──────────────────────────────────────────────────────────────────────
print("Calculando UMAP (n_neighbors=30, min_dist=0.1)...")
reducer_umap = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=2, random_state=42, verbose=False)
X_umap = reducer_umap.fit_transform(X_pca)
print("UMAP OK")

In [ ]:
# ── t-SNE ──────────────────────────────────────────────────────────────────────
print("Calculando t-SNE (perplexity=40)...")
reducer_tsne = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, verbose=0)
X_tsne = reducer_tsne.fit_transform(X_pca)
print("t-SNE OK")

In [ ]:
# ── Fig 1: UMAP coloreado por temperatura (gradiente continuo) ────────────────
temps_unique = sorted(np.unique(y_temp))
cmap = plt.get_cmap("plasma")
norm = plt.Normalize(y_temp.min(), y_temp.max())

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(X_umap[:,0], X_umap[:,1], c=y_temp, cmap="plasma",
                s=8, alpha=0.6, linewidths=0)
cbar = plt.colorbar(sc, ax=ax, label="Temperatura [°C]", fraction=0.035)
ax.set_xlabel("UMAP 1", fontsize=10); ax.set_ylabel("UMAP 2", fontsize=10)
ax.set_title("UMAP — espacio de features coloreado por temperatura",
             fontsize=12, fontweight="bold")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("umap_temperatura_gradiente.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 2: UMAP coloreado por rango térmico ───────────────────────────────────
range_colors = {"frio": "#2E75B6", "templado": "#70AD47", "caliente": "#C00000"}
fig, ax = plt.subplots(figsize=(10, 8))
for rng, color in range_colors.items():
    mask = y_rng == rng
    ax.scatter(X_umap[mask,0], X_umap[mask,1], c=color, s=8, alpha=0.5,
               label=f"{rng} ({mask.sum()})", linewidths=0)
ax.set_xlabel("UMAP 1", fontsize=10); ax.set_ylabel("UMAP 2", fontsize=10)
ax.set_title("UMAP — coloreado por rango térmico (frío / templado / caliente)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10, markerscale=3); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("umap_rangos_termicos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 3: UMAP por familia de features (subplots) ────────────────────────────
FAMILIES_DEF = {
    "Magnitud" : [f for f in FEAT_COLS if f.startswith("mag_")],
    "Fase"     : [f for f in FEAT_COLS if f.startswith("phase_")],
    "PDP"      : [f for f in FEAT_COLS if f.startswith("pdp_")],
    "SVD"      : [f for f in FEAT_COLS if f.startswith("svd_")],
    "Doppler"  : [f for f in FEAT_COLS if f.startswith("doppler_")],
    "dH/dt"    : [f for f in FEAT_COLS if f.startswith("dH_dt_")],
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (fam_name, fam_feats) in enumerate(FAMILIES_DEF.items()):
    if not fam_feats:
        axes[i].set_visible(False); continue
    X_fam = scaler.fit_transform(imp.fit_transform(df_sub[fam_feats].values))
    n_pca = min(min(X_fam.shape)-1, 20)
    X_fam_pca = PCA(n_components=n_pca, random_state=42).fit_transform(X_fam)
    X_fam_umap = umap.UMAP(n_neighbors=20, min_dist=0.15, n_components=2,
                            random_state=42, verbose=False).fit_transform(X_fam_pca)
    sc = axes[i].scatter(X_fam_umap[:,0], X_fam_umap[:,1], c=y_temp, cmap="plasma",
                         s=8, alpha=0.6, linewidths=0)
    plt.colorbar(sc, ax=axes[i], fraction=0.04, label="°C")
    axes[i].set_title(f"{fam_name} ({len(fam_feats)} features)", fontsize=10, fontweight="bold")
    axes[i].set_xlabel("UMAP 1", fontsize=8); axes[i].set_ylabel("UMAP 2", fontsize=8)
    axes[i].grid(alpha=0.2)
    print(f"{fam_name} OK")

plt.suptitle("UMAP por familia de features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("umap_por_familia.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 4: UMAP vs t-SNE comparativa ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (X_red, title) in zip(axes, [(X_umap, "UMAP"), (X_tsne, "t-SNE")]):
    sc = ax.scatter(X_red[:,0], X_red[:,1], c=y_temp, cmap="plasma",
                    s=8, alpha=0.6, linewidths=0)
    plt.colorbar(sc, ax=ax, label="Temperatura [°C]", fraction=0.035)
    ax.set_xlabel(f"{title} 1", fontsize=10); ax.set_ylabel(f"{title} 2", fontsize=10)
    ax.set_title(f"{title} — todas las features", fontsize=11, fontweight="bold")
    ax.grid(alpha=0.2)
plt.suptitle("UMAP vs t-SNE: ¿existe estructura de gradiente en el espacio de features?",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("umap_vs_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fig 5: etiqueta individual por temperatura (UMAP) ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))
cmap20 = cm.get_cmap("tab20", len(temps_unique))
for idx, t in enumerate(temps_unique):
    mask = y_temp == t
    ax.scatter(X_umap[mask,0], X_umap[mask,1],
               color=cmap20(idx), s=10, alpha=0.6, label=f"{int(t)}°C", linewidths=0)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7, ncol=1, markerscale=2)
ax.set_xlabel("UMAP 1", fontsize=10); ax.set_ylabel("UMAP 2", fontsize=10)
ax.set_title("UMAP — una clase por temperatura (20 clases)", fontsize=12, fontweight="bold")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig("umap_20clases.png", dpi=150, bbox_inches="tight")
plt.show()